In [89]:
import re
import sqlite3
import tkinter as tk
from tkinter import messagebox
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.pdfgen import canvas
import pandas as pd
import os

# 1) Create Database:

In [3]:
conn = sqlite3.connect('hospital.db')
cursor = conn.cursor()

cursor.execute('''
    CREATE TABLE IF NOT EXISTS Doctors (
        id INTEGER PRIMARY KEY,
        name TEXT,
        email TEXT,
        age INTEGER,
        phone TEXT,
        specialty TEXT,
        password TEXT
    )''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS Patients (
        id INTEGER PRIMARY KEY,
        name TEXT,
        email TEXT,
        age INTEGER,
        phone TEXT,
        disease TEXT,
        doctor_id INTEGER,
        FOREIGN KEY(doctor_id) REFERENCES Doctors(id)
    )''')
conn.commit()

# 2) Create Doctor Class:

In [155]:
class Person:

    def __init__(self, name=None, email=None, age=None, phone=None):
        self.name = name
        self.email = email
        self.age = age
        self.phone = phone

    @staticmethod
    def validate_email(email):
        return re.match(r'^[\w\.-]+@[\w\.-]+\.\w+$', email)

    @staticmethod
    def validate_phone(phone):
        return re.match(r'^\d{10}$', phone)

    @staticmethod
    def validate_name(name):
        return re.match(r'^[A-Za-z ]+$', name)
        8
class Doctor(Person):
    def __init__(self, name=None, email=None, age=None, phone=None, specialty=None, password=None):
        super().__init__(name, email, age, phone)
        self.specialty = specialty
        self.password = password  # No hashing
        
    def add_doctor(self): 
        conn = sqlite3.connect('hospital.db')
        cursor = conn.cursor()
        if not all([Person.validate_name(self.name), Person.validate_email(self.email), Person.validate_phone(self.phone)]):
            print("Invalid doctor data")
            return
        cursor.execute('''
            INSERT INTO Doctors (name, email, age, phone, specialty, password)
            VALUES (?, ?, ?, ?, ?, ?)
        ''', (self.name, self.email, self.age,self.phone, self.specialty, self.password))
        conn.commit()
        print("The doctor has been successfully registered.")
        conn.close()

    def update_doctor(self,doctor_id,name):
        if not all([ Person.validate_name(name)]):
            print("Invalid doctor data")
            return
        conn = sqlite3.connect('hospital.db')
        cursor = conn.cursor()
        cursor.execute('''
            UPDATE Doctors SET  name=? WHERE id=?
        ''', ( name, doctor_id))
        conn.commit()
        print("The doctor has been updated successfully.")
        conn.close()
        
    def delete_doctor(self,doctor_id):
        conn = sqlite3.connect('hospital.db')
        cursor = conn.cursor()
        cursor.execute("DELETE FROM Doctors WHERE id = ?", (doctor_id,))
        conn.commit()
        print("The doctor's information has been successfully deleted.")
        conn.close()
        
    def search_doc_speci(self,specialty):
        conn = sqlite3.connect("hospital.db")
        cursor = conn.cursor()
        rows = cursor.execute("SELECT * FROM Doctors WHERE specialty=?", (specialty,))
        print("View doctor information by specialty")
        for doc in rows:
            print(doc)
        conn.close()
        return rows

    def get_patient_counts(self):
        conn = sqlite3.connect("hospital.db")
        cursor = conn.cursor()
        rows = cursor.execute('''
            SELECT Doctors.name, COUNT(Patients.id) AS patient_count
            FROM Doctors LEFT JOIN Patients ON Doctors.id = Patients.doctor_id
            GROUP BY Doctors.id
        ''')
        print("Number of patients per doctor")
        for doc in rows:
            print(doc)

        conn.close()
        return rows

    def view_doctors(self):
        conn = sqlite3.connect('hospital.db')
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM Doctors")
        rows = cursor.fetchall()
        print("View information about each doctor")
        for doc in rows:
            print(doc)
        conn.close()
        return rows

    def doctor_exists(self,doctor_id):
        conn = sqlite3.connect("hospital.db")
        cursor = conn.cursor()
        cursor.execute("SELECT 1 FROM Doctors WHERE id = ?", (doctor_id,))
        result = cursor.fetchone()
        conn.close()
        return result is not None  
    
    def info_doctors(self):
        conn = sqlite3.connect('hospital.db')
        cursor = conn.cursor()
        cursor.execute("SELECT id,name,specialty FROM Doctors")
        rows = cursor.fetchall()
        print("Doctors available in the hospital")
        for doc in rows:
            print(doc)
        conn.close()
        return rows
    
    def get_unique_specialties(self):
        conn = sqlite3.connect('hospital.db')
        cursor = conn.cursor()  
        cursor.execute("SELECT DISTINCT specialty FROM Doctors")
        specialties = [row[0] for row in cursor.fetchall()]
        for specialtie in specialties:
            print(specialtie, end=" ")
        conn.close()
        return specialties
    
    def count_patients_for_doctor(self,doctor_id):
        conn = sqlite3.connect("hospital.db")
        cursor = conn.cursor()
        cursor.execute("SELECT COUNT(*) FROM Patients WHERE doctor_id = ?", (doctor_id,))
        result = cursor.fetchone()[0]
        conn.close()
        if result > 0:
            print(f"There are {result} patient(s) assigned to doctor with ID {doctor_id}.")
            print("It cannot be deleted.")
            return True
    
    def doctor_login(self):
        email = email_entry.get()
        password = password_entry.get()
        if not email or not password:
            messagebox.showwarning("warning!!", "Please fill in all fields.")
            return
        conn = sqlite3.connect('hospital.db')
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM Doctors WHERE email=? AND password=?", (email, password))
        doctor = cursor.fetchone()
        conn.close()
        if doctor:
            messagebox.showinfo("نجاح", f"مرحبًا دكتور {doctor[1], doctor[3],doctor[4], doctor[4]}!")
        else:
            messagebox.showerror("mistake!!", "Incorrect email or password.")

    def exportdoctorpdf(self):
        # الاتصال بقاعدة البيانات
        conn = sqlite3.connect("hospital.db")
        cursor = conn.cursor()

        # جلب قائمة الأطباء مع عدد المرضى لديهم
        cursor.execute("""
        SELECT d.id, d.name, d.specialty, 
               COUNT(p.id) AS patient_count
        FROM Doctors d
        LEFT JOIN Patients p ON d.id = p.doctor_id
        GROUP BY d.id
        """)
    
        doctors_data = cursor.fetchall()
        conn.close()

        data = [['Doctor ID', 'Name', 'Specialty', 'Number of Patients']]
        for doctor in doctors_data:
            doctor_id, name, specialty, patient_count = doctor
            if patient_count is None:
                patient_count = 'null'

            data.append([doctor_id, name, specialty, patient_count])
        pdf_filename = "doctors_report.pdf"
        pdf = SimpleDocTemplate(pdf_filename, pagesize=letter)
        table = Table(data)
    
        style = TableStyle([
            ('BACKGROUND', (0, 0), (-1, 0), colors.grey),
            ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
            ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
            ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
            ('BOTTOMPADDING', (0, 0), (-1, 0), 12),
            ('BACKGROUND', (0, 1), (-1, -1), colors.beige),
            ('GRID', (0, 0), (-1, -1), 1, colors.black),
        ])
    
        table.setStyle(style)
        elements = [table]
        pdf.build(elements)
        print(f"PDF generated: {pdf_filename}")

    
    
    def exportdoctorexcel(self):
        conn = sqlite3.connect("hospital.db")
        cursor = conn.cursor()

        cursor.execute("""
            SELECT d.id, d.name, d.specialty, 
                   COUNT(p.id) AS patient_count
            FROM Doctors d
            LEFT JOIN Patients p ON d.id = p.doctor_id
            GROUP BY d.id
        """)

        doctors_data = cursor.fetchall()
        conn.close()

        # إنشاء DataFrame من البيانات
        df = pd.DataFrame(doctors_data, columns=['Doctor ID', 'Name', 'Specialty', 'Number of Patients'])

        # تحديد مسار حفظ الملف في نفس مجلد العمل الحالي
        folder_path = os.getcwd()
        excel_filename = os.path.join(folder_path, "doctors_report.xlsx")

        # حفظ إلى ملف Excel
        df.to_excel(excel_filename, index=False)
        print(f"Excel file saved at: {excel_filename}")




    def User_Interface(self):
        doctor = Doctor()
        global email_entry, password_entry
        root = tk.Tk()
        root.title("تسجيل دخول الطبيب")
        root.geometry("350x250")
        root.configure(bg="#f0f8ff")  
        frame = tk.Frame(root, bg="#f8faff", bd=2, relief="ridge")
        frame.place(relx=0.5, rely=0.5, anchor="center")
        tk.Label(frame, text="LogIn", font=("Arial", 14, "bold"), bg="#f8faff", fg="#003366").grid(row=0, column=0, columnspan=2, pady=10)
        tk.Label(frame, text="Email:", bg="#f8faff", anchor="w").grid(row=1, column=0, sticky="e", padx=10, pady=5)
        email_entry = tk.Entry(frame, width=25)
        email_entry.grid(row=1, column=1, pady=5)
        tk.Label(frame, text="Password:", bg="#f8faff", anchor="w").grid(row=2, column=0, sticky="e", padx=10, pady=5)
        password_entry = tk.Entry(frame, width=25, show="*")
        password_entry.grid(row=2, column=1, pady=5)
        tk.Button(frame, text="Register", bg="#0066cc", fg="white", width=20, command=doctor.doctor_login).grid(row=3, column=0, columnspan=2, pady=15)
        root.mainloop()

# 3) Create Patient Class:

In [197]:
class Patient(Person):
    def __init__(self, name=None, email=None, age=None, phone=None, disease=None, doctor_id=None):
        super().__init__(name, email, age, phone)
        self.disease = disease
        self.doctor_id = doctor_id

    def add_patient(self,name, email, age, phone, disease, doctor_id=None):
        if not all([Person.validate_name(name), Person.validate_email(email), Person.validate_phone(phone)]):
            print("Invalid patient data")
            return
        conn = sqlite3.connect("hospital.db")
        cursor = conn.cursor()
        cursor.execute('''
            INSERT INTO Patients (name, email, age, phone, disease, doctor_id)
            VALUES (?, ?, ?, ?, ?, ?)
        ''', (name, email, age, phone, disease, doctor_id))
        conn.commit()
        print("The Patients has been successfully registered.")
        conn.close()

    def update_patient(self,patient_id,name, doctor_id):
        if not all([Person.validate_name(name)]):
            print("Invalid patient data")
            return
        conn = sqlite3.connect("hospital.db")
        cursor = conn.cursor()
        cursor.execute('''
            UPDATE Patients SET  name=?, doctor_id=? WHERE id=?
        ''', ( name, doctor_id, patient_id))
        conn.commit()
        print("The Patients has been updated successfully.")
        conn.close()

    def delete_patient(self,patient_id):
        conn = sqlite3.connect("hospital.db")
        cursor = conn.cursor()
        cursor.execute("DELETE FROM Patients WHERE id = ?", (patient_id,))
        conn.commit()
        print("The Patient's information has been successfully deleted.")
        conn.close()

    def search_patients_by_disease(self,disease):
        conn = sqlite3.connect("hospital.db")
        cursor = conn.cursor()
        patients = cursor.execute("SELECT * FROM Patients WHERE disease LIKE ?", ('%' + disease + '%',))
        print("View Patients information by disease.")
        for patient in patients:
            print(patient)
        conn.close()
        return patients
    
    def view_patients(self):
        conn = sqlite3.connect("hospital.db")
        cursor = conn.cursor()
        cursor = cursor.execute("SELECT * FROM Patients")
        patients = cursor.fetchall()
        print("View information about each patients")
        for patient in patients:
            print(patient, end=" ")
            print()
        conn.close()
        return patients

    def patient_exists(self,patient_id):
        conn = sqlite3.connect("hospital.db")
        cursor = conn.cursor()
        cursor.execute("SELECT 1 FROM Patients WHERE id = ?", (patient_id,))
        result = cursor.fetchone()
        conn.close()
        return result is not None

    def exportpatientspdf(self):
        # الاتصال بقاعدة البيانات
        conn = sqlite3.connect("hospital.db")
        cursor = conn.cursor()

        # استعلام للحصول على معلومات المرضى مع معلومات الطبيب (باستخدام LEFT JOIN لدعم المرضى الذين ليس لديهم طبيب)
        cursor.execute("""
            SELECT 
                Patients.id, 
                Patients.name, 
                Patients.age, 
                Patients.disease, 
                IFNULL(Doctors.name, 'No Doctor Assigned') AS doctor_name,
        IFNULL(Doctors.specialty, 'No Specialty') AS doctor_specialty
            FROM Patients
            LEFT JOIN Doctors ON Patients.doctor_id = Doctors.id
        """)
    
        rows = cursor.fetchall()
        conn.close()

        # إضافة رؤوس الأعمدة
        data = [['ID', 'Name', 'Age', 'Disease', 'Doctor Name', 'Doctor Specialty']] + rows

        # إنشاء ملف PDF
        pdf = SimpleDocTemplate("patients_report.pdf", pagesize=letter)
        table = Table(data)

        # تنسيق الجدول في PDF
        style = TableStyle([
            ('BACKGROUND', (0, 0), (-1, 0), colors.grey),
            ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
            ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
            ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
            ('BOTTOMPADDING', (0, 0), (-1, 0), 12),
            ('BACKGROUND', (0, 1), (-1, -1), colors.beige),
            ('GRID', (0, 0), (-1, -1), 1, colors.black),
        ])
        table.setStyle(style)

        # إضافة الجدول إلى مستند PDF
        elements = [table]
        pdf.build(elements)
        print("PDF saved successfully.")

    
    def exportpatientsexcel(self):
        # الاتصال بقاعدة البيانات
        conn = sqlite3.connect("hospital.db")
        cursor = conn.cursor()

        # استعلام للحصول على معلومات المرضى مع معلومات الطبيب (باستخدام LEFT JOIN لدعم المرضى الذين ليس لديهم طبيب)
        cursor.execute("""
            SELECT 
                Patients.id, 
                Patients.name, 
                Patients.age, 
                Patients.disease, 
                IFNULL(Doctors.name, 'No Doctor Assigned') AS doctor_name,
                IFNULL(Doctors.specialty, 'No Specialty') AS doctor_specialty
            FROM Patients
            LEFT JOIN Doctors ON Patients.doctor_id = Doctors.id
        """)
    
        rows = cursor.fetchall()
        conn.close()

        # تحويل البيانات إلى DataFrame
        df = pd.DataFrame(rows, columns=['ID', 'Name', 'Age', 'Disease', 'Doctor Name', 'Doctor Specialty'])

        # تحديد المسار الكامل لحفظ الملف
        folder_path = os.getcwd()  # مجلد العمل الحالي
        excel_filename = os.path.join(folder_path, "patients_report.xlsx")
        df.to_excel(excel_filename, index=False)

        print(f"Excel file saved successfully at: {excel_filename}")
    def close_connection():
        conn.close()

    


In [199]:
def validate_all(name, email, phone):
    if not re.match(r'^[A-Za-z ]+', name):
        print("❌ الاسم غير صالح. يجب أن يحتوي فقط على أحرف ومسافات.")
        return False
    if not re.match(r'^[-̇]+@[-̇]++', email):
        print("❌ البريد الإلكتروني غير صالح.")
        return False
    if not re.match(r'^1̣0̣$', phone):
        print("❌ رقم الهاتف غير صالح. يجب أن يتكون من 10 أرقام.")
        return False
    print("✅ جميع القيم صحيحة.")
    return True
 

# 4) Code implementation:

In [206]:
def omar():
    doctors = []
    patient = []
    doctor = Doctor()
    patient = Patient()
    print("Welcome to our Hospital Management System!")
    print('''
      Enter 1 Add a new doctor.
      Enter 2 Update doctor information.
      Enter 3 Delete doctor information.
      Enter 4 Search for a doctor by specialty.
      Enter 5 Knowing the number of patients per doctor.
      Enter 6 View doctors' information.
      Enter 7 I am a doctor Login.
      
      Enter 8 Add a patient.
      Enter 9 Update patient information.
      Enter 10 Delete patient information.
      Enter 11 Search for patients with a specific disease.
      Enter 12 View all patient information.

          
      Enter 13 To export report for doctor:Pdf
      Enter 14 To export report for pationt:Pdf
      Enter 15 To export report for doctor:Excel
      Enter 16 To export report for pationt:Exel

      Enter 17 to exit
      

          ''')
    while(True):
      number = int(input("Please Enter a service number: "))
      print()
      if number == 1:
          name = input("Please, Enter Doctor's name: ")
          age = int(input("Please, Enter Doctor's age: "))
          phone = input("Please, Enter Doctor's phone: ")
          specialty = input("Please, Enter Doctor's specialty: ")
          email = input("Please, Enter Doctor's email: ")
          password = input("Please, Enter New Doctor's password: ")
          doctor = Doctor(name, email, age, phone, specialty, password)
          doctor.add_doctor()

      elif number == 2: 
          ID = int(input("Enter Doctor's ID: "))
          name = input("Please, Enter New Doctor's name: ")
          #age = int(input("Please, Enter New Doctor's age: "))
          #phone = input("Please, Enter New Doctor's phone: ")
          #specialty = input("Please, Enter New Doctor's specialty: ")
          #email = input("Please, Enter New Doctor's email: ")
          #password = input("Please, Enter New Doctor's password: ")
          if doctor.doctor_exists(ID):
              doctor.update_doctor(ID,name)
          else:
              print("Doctor is not exists")
          

      elif number == 3:   
          ID = int(input("Enter Doctor's ID: "))
          if not doctor.count_patients_for_doctor(ID):
              if doctor.doctor_exists(ID):
                  doctor.delete_doctor(ID)
              else:
                  print("Doctor is not exists")
                 
      elif number == 4:
          print("Specializations available in the hospital:")
          doctor.get_unique_specialties()
          specialty = input("Please, Enter Doctor's specialty: ")
          doctor.search_doc_speci(specialty)
          
      elif number == 5:  
           doctor.get_patient_counts()

      elif number == 6:        
           doctor.view_doctors()
          
      elif number == 7:
           doctor.User_Interface()

      elif number == 8:
          name = input("Please, Enter Patient's name: ")
          age = int(input("Please, Enter Patient's age: "))
          phone = input("Please, Enter Patient's phone: ")
          disease = input("Please, Enter Patient's disease: ")
          email = input("Please, Enter Patient's email: ")
          password = input("Please, Enter Patient's password: ")
          doctor.info_doctors()
          ID = int(input("Enter Doctor's ID: "))
          if  doctor.doctor_exists(ID):
              patient1 = Patient(name, email, age, phone, disease, ID)
              patient.add_patient(patient1.name, patient1.email, age, phone, patient1.disease, patient1.doctor_id)
          else:
              print("Doctor is not exists")
    
      elif number == 9:
          id = int(input("Please, Enter Patient's ID: "))
          name = input("Please, Enter Patient's name: ")
          #age = int(input("Please, Enter Patient's age: "))
          #phone = input("Please, Enter Patient's phone: ")
          #disease = input("Please, Enter Patient's disease: ")
          #email = input("Please, Enter Patient's email: ")
          #password = input("Please, Enter Patient's password: ")
          doctor.info_doctors()
          ID = int(input("Enter Doctor's ID: "))
          if doctor.doctor_exists(ID) and patient.patient_exists(id):
              patient.update_patient(id,name, ID)
          else:
              print("Doctor or Patient is not exists")

      elif number == 10:
          ID = int(input("Enter Patient's ID: "))
          if patient.patient_exists(ID):
              patient.delete_patient(ID)
          else:
              print("Patient is not exists")
          
      elif number == 11:
          disease = input("Please, Enter Patient's disease: ")
          patient.search_patients_by_disease(disease)

      elif number == 12:
          patient.view_patients()

      elif number == 13:
          doctor.exportdoctorpdf()
          
      elif number == 14:
          patient.exportpatientspdf()

      elif number == 15:
          doctor.exportdoctorexcel()
          
      elif number == 16:
          patient.exportpatientsexcel()

      elif number == 17:
        print("Thanks for using our management system! Bye :)\n")
        break

      else:
        print("Please Enter a valid number.\n")


##### 

In [ ]:
 omar()

Welcome to our Hospital Management System!

      Enter 1 Add a new doctor.
      Enter 2 Update doctor information.
      Enter 3 Delete doctor information.
      Enter 4 Search for a doctor by specialty.
      Enter 5 Knowing the number of patients per doctor.
      Enter 6 View doctors' information.
      Enter 7 I am a doctor Login.
      
      Enter 8 Add a patient.
      Enter 9 Update patient information.
      Enter 10 Delete patient information.
      Enter 11 Search for patients with a specific disease.
      Enter 12 View all patient information.

          
      Enter 13 To export report for doctor:Pdf
      Enter 14 To export report for pationt:Pdf
      Enter 15 To export report for doctor:Excel
      Enter 16 To export report for pationt:Exel

      Enter 17 to exit
      

          


Please Enter a service number:  4



Specializations available in the hospital:
it heart 

Please, Enter Doctor's specialty:  heart


View doctor information by specialty
(3, 'mike', 'mo5rb600@mail.com', 22, '0948888888', 'heart', '12345')


...


PDF generated: doctors_report.pdf


Please Enter a service number:  14



PDF saved successfully.


Please Enter a service number:  15



Excel file saved at: C:\Users\aa\Desktop\pl\doctors_report.xlsx


In [12]:
pip install reportlab


Note: you may need to restart the kernel to use updated packages.
